# Customer Churn Prediction

This notebook explores customer churn in the Telco dataset and builds machine learning models to predict whether a customer is likely to leave.

The workflow includes data inspection, exploratory data analysis, data cleaning, preprocessing, model training, evaluation, and hyperparameter tuning.

In [ ]:
# Import the libraries used in this notebook
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    make_scorer
)

## 1. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns


In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True) * 100


In [ ]:
df.describe()

### 2.1 Overall Churn Distribution

In [ ]:

sns.countplot(data=df, x="Churn")

plt.title("Customer Churn Distribution")
plt.xlabel("Customer Churn")
plt.ylabel("Number of Customers")

plt.show()

### 2.2 Contract Type

In [ ]:
sns.countplot(data=df, x="Contract", hue="Churn")

plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

contract_churn

In [ ]:
contract_churn["Yes"].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)

plt.show()

### 2.3 Customer Tenure

In [ ]:
sns.boxplot(data=df, x="Churn", y="tenure")

plt.title("Customer Tenure vs Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure (Months)")

plt.show()

In [ ]:
df.groupby("Churn")["tenure"].mean()

In [ ]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 72],
    labels=["0-12 Months", "13-24 Months", "25+ Months"]
)

In [ ]:
tenure_churn = pd.crosstab(
    df["TenureGroup"],
    df["Churn"],
    normalize="index"
) * 100

tenure_churn

In [ ]:
tenure_churn["Yes"].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)

plt.show()

### 2.4 Monthly Charges

In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")

plt.title("Monthly Charges vs Customer Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

plt.show()

In [ ]:
df.groupby("Churn")["MonthlyCharges"].mean()

In [ ]:
avg_charges = df.groupby("Churn")["MonthlyCharges"].mean()

avg_charges.plot(kind="bar", figsize=(7, 4))

plt.title("Average Monthly Charges by Churn")
plt.xlabel("Churn")
plt.ylabel("Average Monthly Charges")
plt.xticks(rotation=0)

plt.show()

### 2.5 Internet Service

In [ ]:
sns.countplot(data=df, x="InternetService", hue="Churn")

plt.title("Customer Churn by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

internet_churn

In [ ]:
internet_churn["Yes"].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)

plt.show()

### 2.6 Payment Method

In [ ]:
sns.countplot(
    data=df,
    x="PaymentMethod",
    hue="Churn"
)

plt.title("Customer Churn by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Customers")
plt.xticks(rotation=25, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
payment_churn = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
) * 100

payment_churn

In [ ]:
payment_churn["Yes"].plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=25, ha="right")

plt.tight_layout()
plt.show()

### 2.7 Online Security and Tech Support

In [ ]:
sns.countplot(data=df, x="OnlineSecurity", hue="Churn")

plt.title("Customer Churn by Online Security")
plt.xlabel("Online Security")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(data=df, x="TechSupport", hue="Churn")

plt.title("Customer Churn by Tech Support")
plt.xlabel("Tech Support")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
for col in ["OnlineSecurity", "TechSupport"]:
    churn_rate = pd.crosstab(
        df[col],
        df["Churn"],
        normalize="index"
    ) * 100

    print(f"\nChurn rate by {col}:")
    print(churn_rate)

### 2.8 Senior Citizen Status

In [ ]:
sns.countplot(data=df, x="SeniorCitizen", hue="Churn")

plt.title("Customer Churn by Senior Citizen Status")
plt.xlabel("Senior Citizen (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()


In [ ]:
senior_churn = pd.crosstab(
    df["SeniorCitizen"],
    df["Churn"],
    normalize="index"
) * 100

senior_churn

### 2.9 EDA Summary

In [ ]:
eda_summary = pd.DataFrame({
    "Feature": [
        "Contract",
        "tenure",
        "MonthlyCharges",
        "InternetService",
        "PaymentMethod",
        "OnlineSecurity",
        "TechSupport",
        "SeniorCitizen"
    ],
    "Analysis": [
        "Contract type vs churn rate",
        "Tenure distribution and churn rate",
        "Monthly charges vs churn",
        "Internet service vs churn rate",
        "Payment method vs churn rate",
        "Online security vs churn",
        "Technical support vs churn",
        "Senior citizen status vs churn"
    ]
})

eda_summary

## 3. Data Cleaning


In [ ]:
df_clean = df.copy()

In [ ]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

In [ ]:
df_clean["TotalCharges"].isnull().sum()

In [ ]:
df_clean.loc[
    df_clean["TotalCharges"].isnull(),
    ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

In [ ]:
df_clean = df_clean.dropna(subset=["TotalCharges"])

In [ ]:
df_clean["TotalCharges"].isnull().sum()

In [ ]:
df_clean.isnull().sum().sum()

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean["customerID"].nunique()

In [ ]:
df_clean = df_clean.drop(columns=["TenureGroup"])

In [ ]:
df_clean.shape


In [ ]:
df_clean.info()

## 4. Feature Preparation



In [ ]:
X = df_clean.drop(columns=["Churn", "customerID"])
y = df_clean["Churn"]

In [ ]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

In [ ]:
print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

## 5. Train-Test Split and Preprocessing


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
print("Training churn distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True) * 100)

### 5.1 Build the Preprocessing Transformer


In [ ]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 6. Train Baseline Models

In [ ]:


logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [ ]:
logistic_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_pipeline = logistic_pipeline.predict(X_test)

In [ ]:


print(
    classification_report(
        y_test,
        y_pred_pipeline
    )
)

### 6.1 Decision Tree

In [ ]:
dt_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ))
    ]
)

In [ ]:
dt_pipeline.fit(X_train,y_train)

In [ ]:
y_pred_dt = dt_pipeline.predict(X_test)

In [ ]:
dt_accuracy = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", dt_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_dt
)

plt.title("Decision Tree - Confusion Matrix")
plt.show()

### 6.2 Random Forest

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
rf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf
)

plt.title("Random Forest - Confusion Matrix")
plt.show()

## 7. Hyperparameter Tuning


In [ ]:
param_grid = {
    "classifier__max_depth": [3, 5, 7, 10, 15, None],
    "classifier__min_samples_split": [2, 10, 20],
    "classifier__min_samples_leaf": [1, 5, 10]
}

In [ ]:
from sklearn.metrics import f1_score, make_scorer

f1_yes_scorer = make_scorer(
    f1_score,
    pos_label="Yes"
)

### 7.1 Tune the Decision Tree

In [ ]:
dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    scoring=f1_yes_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1,
    error_score="raise"
)

In [ ]:
dt_grid.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(dt_grid.best_params_)

print("\nBest Cross-Validation F1-Score:")
print(dt_grid.best_score_)

In [ ]:
best_dt_model = dt_grid.best_estimator_

In [ ]:
y_pred_best_dt = best_dt_model.predict(X_test)

In [ ]:
print("Tuned Decision Tree Accuracy:")
print(accuracy_score(y_test, y_pred_best_dt))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_best_dt
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_dt
)

plt.title("Tuned Decision Tree - Confusion Matrix")
plt.show()

In [ ]:
original_dt_f1 = f1_score(
    y_test,
    y_pred_dt,
    pos_label="Yes"
)

tuned_dt_f1 = f1_score(
    y_test,
    y_pred_best_dt,
    pos_label="Yes"
)

print("Original Decision Tree F1:", original_dt_f1)
print("Tuned Decision Tree F1:", tuned_dt_f1)

### 7.2 Tune Logistic Regression

In [ ]:
lr_param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

In [ ]:
lr_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=lr_param_grid,
    scoring=f1_yes_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1,
    error_score="raise"
)

In [ ]:
lr_grid.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(lr_grid.best_params_)

print("\nBest Cross-Validation F1-Score:")
print(lr_grid.best_score_)

In [ ]:
best_lr_model = lr_grid.best_estimator_

In [ ]:
y_pred_best_lr = best_lr_model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_best_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_lr))

yes_index = list(best_lr_model.classes_).index("Yes")
y_prob_best_lr = best_lr_model.predict_proba(X_test)[:, yes_index]

print(
    "ROC-AUC:",
    roc_auc_score(
        (y_test == "Yes").astype(int),
        y_prob_best_lr
    )
)

In [ ]:


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best_lr
)

plt.title("Tuned Logistic Regression - Confusion Matrix")
plt.show()

In [ ]:
original_lr_f1 = f1_score(
    y_test,
    logistic_pipeline.predict(X_test),
    pos_label="Yes"
)

tuned_lr_f1 = f1_score(
    y_test,
    y_pred_best_lr,
    pos_label="Yes"
)

print("Original Logistic Regression F1:", original_lr_f1)
print("Tuned Logistic Regression F1:", tuned_lr_f1)

## 8. Final Model Comparison

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    yes_index = list(model.classes_).index("Yes")
    y_prob = model.predict_proba(X_test)[:, yes_index]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, pos_label="Yes"
        ),
        "Recall": recall_score(
            y_test, y_pred, pos_label="Yes"
        ),
        "F1-Score": f1_score(
            y_test, y_pred, pos_label="Yes"
        ),
        "ROC-AUC": roc_auc_score(
            (y_test == "Yes").astype(int),
            y_prob
        )
    }

In [ ]:
models = [
    ("Logistic Regression", logistic_pipeline),
    ("Tuned Logistic Regression", best_lr_model),
    ("Decision Tree", dt_pipeline),
    ("Tuned Decision Tree", best_dt_model),
    ("Random Forest", rf_pipeline)
]

results = []

for name, model in models:
    results.append(
        evaluate_model(name, model, X_test, y_test)
    )

final_comparison = pd.DataFrame(results)

final_comparison.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)